In [1]:
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import time

In [2]:
# load the data
df = pd.read_csv('../../data/cleaned/wednesday_cleaned.csv')
print(f'loaded data: {df.shape}')

loaded data: (61001, 69)


In [3]:
# separate features and target
X = df.drop(['Label', 'Attack'], axis=1)
y = df['Attack']

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')

X shape: (61001, 67)
y shape: (61001,)


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# using same split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# apply scaling
scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('data ready')
print(f'train: {X_train_scaled.shape}')
print(f'test: {X_test_scaled.shape}')

data ready
train: (48800, 67)
test: (12201, 67)


## Naive Bayes

gaussian NB since our features are continuous

In [5]:
# train naive bayes
start = time.time()
nb_model = GaussianNB()
nb_model.fit(X_train_scaled, y_train)
train_time = time.time() - start

print(f'training time: {train_time:.2f}s')

training time: 0.03s


In [6]:
# check performance
y_pred_nb = nb_model.predict(X_test_scaled)
acc_nb = accuracy_score(y_test, y_pred_nb)
print(f'test accuracy: {acc_nb:.4f}')
print('\nconfusion matrix:')
print(confusion_matrix(y_test, y_pred_nb))
print('\nclassification report:')
print(classification_report(y_test, y_pred_nb))

test accuracy: 0.9826

confusion matrix:
[[  924    75]
 [  137 11065]]

classification report:
              precision    recall  f1-score   support

           0       0.87      0.92      0.90       999
           1       0.99      0.99      0.99     11202

    accuracy                           0.98     12201
   macro avg       0.93      0.96      0.94     12201
weighted avg       0.98      0.98      0.98     12201



## k-Nearest Neighbors

trying with k=5 

In [6]:
# knn with k=5
start = time.time()
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_scaled, y_train)
train_time = time.time() - start

print(f'training time: {train_time:.2f}s')

training time: 0.01s


In [8]:
# evaluate knn
y_pred_knn = knn_model.predict(X_test_scaled)
acc_knn = accuracy_score(y_test, y_pred_knn)
print(f'test accuracy: {acc_knn:.4f}')
print('\nconfusion matrix:')
print(confusion_matrix(y_test, y_pred_knn))
print('\nclassification report:')
print(classification_report(y_test, y_pred_knn))

test accuracy: 0.9982

confusion matrix:
[[  984    15]
 [    7 11195]]

classification report:
              precision    recall  f1-score   support

           0       0.99      0.98      0.99       999
           1       1.00      1.00      1.00     11202

    accuracy                           1.00     12201
   macro avg       1.00      0.99      0.99     12201
weighted avg       1.00      1.00      1.00     12201



In [9]:
# trying k=3
knn_k3 = KNeighborsClassifier(n_neighbors=3)
knn_k3.fit(X_train_scaled, y_train)
y_pred_k3 = knn_k3.predict(X_test_scaled)
acc_k3 = accuracy_score(y_test, y_pred_k3)
print(f'k=3 accuracy: {acc_k3:.4f}')
print(f'k=5 accuracy: {acc_knn:.4f}')

k=3 accuracy: 0.9989
k=5 accuracy: 0.9982


## Logistic Regression



In [10]:
# train logistic regression
start = time.time()
lr_model = LogisticRegression(max_iter=1000, solver='liblinear', random_state=42)
lr_model.fit(X_train_scaled, y_train)
train_time = time.time() - start

print(f'training time: {train_time:.2f}s')

training time: 1.84s


In [11]:
# evaluate 
y_pred_lr = lr_model.predict(X_test_scaled)
acc_lr = accuracy_score(y_test, y_pred_lr)
print(f'test accuracy: {acc_lr:.4f}')
print('\nconfusion matrix:')
print(confusion_matrix(y_test, y_pred_lr))
print('\nclassification report:')
print(classification_report(y_test, y_pred_lr))

test accuracy: 0.9937

confusion matrix:
[[  924    75]
 [    2 11200]]

classification report:
              precision    recall  f1-score   support

           0       1.00      0.92      0.96       999
           1       0.99      1.00      1.00     11202

    accuracy                           0.99     12201
   macro avg       1.00      0.96      0.98     12201
weighted avg       0.99      0.99      0.99     12201



In [12]:
# check which knn to save
if acc_k3 > acc_knn:
    final_knn = knn_k3
    print('using k=3 for knn')
else:
    final_knn = knn_model
    print('using k=5 for knn')

using k=3 for knn


In [14]:
save_dir = '../../models/saved_model/'
os.makedirs(save_dir, exist_ok=True)

# save models
with open(os.path.join(save_dir, 'naive_bayes.pkl'), 'wb') as f:
    pickle.dump(nb_model, f)
    
with open(os.path.join(save_dir, 'knn.pkl'), 'wb') as f:
    pickle.dump(final_knn, f)
    
with open(os.path.join(save_dir, 'logistic_regression.pkl'), 'wb') as f:
    pickle.dump(lr_model, f)

print('models saved to', save_dir)

models saved to ../../models/saved_model/


In [15]:
# also need to save the scaler
with open(os.path.join(save_dir, 'scaler.pkl'), 'wb') as f:
    pickle.dump(scaler, f)
print('scaler saved')

scaler saved


### 12/8/2025 - Umar

trained three simple classifiers on the DoS dataset:

**naive bayes**: fast to train but probably not ideal given feature correlations. will see how it compares to other models

**knn**: tested both k=3 and k=5

**logistic regression**: used liblinear solver, converged fine. 

all models saved to models/saved_model/ as .pkl files. also saved the scaler since we need it for preprocessing new data

